In [ ]:
import sys
import os
from datetime import datetime
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pickle
import japanize_matplotlib
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# プロジェクトのルートディレクトリをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.simulation.run_emates import run_parallel_emates_simulations
from src.simulation.data_load import save_data_to_pickle
from src.util.path_manager import get_paths

def total_costs_yearly(result_file) -> tuple:
    """充電ステーションのコスト計算:https://www.notion.so/2098044c3b9180acade1c42c52872eda?source=copy_link
    CSコスト：初期コスト＋運用コスト

    初期コストは以下の三つで構成される。
    1. 充電器本体コスト
    2. 変電設備コスト
    3. 一か所あたりの設置コスト

    運用コストは以下の三つで構成される。
    1. +保守コスト
    2. +電気料金（契約＋従量）
    3. -充電収益
    """
    # コストの定義=======================
    CHARGER_COST = {"50": 380, "90": 666, "100": 730} 
    SUBSTATION_COST_PER_KW = 2 # 万円/kw
    INSTALLATION_COST = 250 # 万円
    # 変数の設定=========================
    # データ読み込み
    with open(result_file, 'rb') as f:
        emates_result = pickle.load(f)
    timeseries_kw = emates_result['time_series_kw']
    cs_config = emates_result['cs_config']
    
    capacity_list = cs_config['cap_kw']
    ports_list = cs_config['ports']
    actual_csids = timeseries_kw.columns.tolist()
    
    vehicle_trip = emates_result['vehicle_trip']
    vehicle_trip['trip_time'] = vehicle_trip['EndTime'] - vehicle_trip['StartTime']

    # ポート数と容量の配列を作成
    ports_array = np.array(ports_list)
    capacity_array = np.array(capacity_list)
    max_capacity = ports_array * capacity_array
    # -------------------------------------------
    # 1.初期コストの概算
    # 1.1：充電器本体コスト
    charger_costs = np.array([CHARGER_COST[str(c)] for c in capacity_list])
    cs_costs = charger_costs * ports_array
    # 1.2：変電設備コスト
    substation_costs = (SUBSTATION_COST_PER_KW * max_capacity).reshape(-1)

    # 1.3：一か所あたりの設置コスト.ポートが0のときは設置コストも0
    isnan_ports = (ports_array == 0)
    installation_costs = np.where(isnan_ports, 0, INSTALLATION_COST)
    total_installation_cost = installation_costs.sum()

    # 1.4：総コストの計算（充電器本体＋キュービクル＋工事コスト）
    initial_costs = np.nan_to_num(cs_costs + substation_costs + installation_costs)

    """運用コスト：
    保守＋電気料金（契約料金＋従量料金ー充電収益）
    CSごとのコストを計算する
    """
    # 2.運用コストの計算========================
    # 2.1： コストの定義
    MAINTENANCE_COST_PER_YEAR = 30 # 万円/年.一か所あたり
    CONTRACT_COST_PER_KW_MONTH = 1911e-4 # 万円/kw
    USAGE_COST_PER_KWH_MONTH = 18e-4 # 万円/kWh
    CHARGING_PRICE_PER_KWH = 50e-4 # 万円/kWh
    
    # 2.2： 年間コスト計算
    maintainance_costs = np.array([MAINTENANCE_COST_PER_YEAR] * len(actual_csids))
    contract_costs = CONTRACT_COST_PER_KW_MONTH * max_capacity.reshape(-1) * 12 # 年間契約料金
    usage_costs = USAGE_COST_PER_KWH_MONTH * timeseries_kw.sum(axis=0) * 12 # 年間従量料金
    charging_revenue = CHARGING_PRICE_PER_KWH * timeseries_kw.sum(axis=0) * 12 # 年間充電収益
    
    # 2.3： 年間総運用コストの計算
    running_costs_yearly = np.nan_to_num(maintainance_costs + contract_costs + usage_costs - charging_revenue)
    
    """ドライバーの総旅行時間の計算"""
    # 3.総旅行時間の計算=======================
    total_trip_time = vehicle_trip['trip_time'].sum() / 3600  # 時間単位に変換
    
    # 初期コスト＋ 年間運用コスト（ベクター）
    total_costs = initial_costs + running_costs_yearly # + total_trip_time * 0.1 # 0.1万円/時間のドライバーコストを追加
    
    return total_costs.sum(), emates_result, total_trip_time, initial_costs.sum(), running_costs_yearly.sum()

In [ ]:
# settings
def setting()-> tuple:
    """設定を行う関数"""
    # SAVEDIRをグローバル変数として初期化。
    current_time = datetime.now().strftime('%Y%m%d_%H%M')
    # SAVE_DIR = current_time
    SAVE_DIR = "../20250703_2245"

    # まず過去データから各指標（充電器本体コスト，総充電量，総旅行時間）の変化を確認。
    # データの読み込み。pklファイルを読み込む。

    data_path = SAVE_DIR
    num_files = len([f for f in os.listdir(data_path) if f.endswith('.pkl')])
    data_files = [f for f in os.listdir(data_path) if f.endswith('.pkl')]
    # ソート
    data_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))  # ファイル名の数字部分でソート
    print(f"読み込んだファイル数: {num_files}")
    return data_files, data_path, num_files

In [ ]:
def get_od_trip_times(vehicle_trip: pd.DataFrame) -> pd.DataFrame:
    """OD別・時間帯別の平均旅行時間を計算する関数"""
    # 充電した車両と充電しなかった車両を分ける
    charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] > 0]
    no_charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] == 0]
    
    charging_trip, no_charging_trip = _add_vehicle_trip_columns(charging_trip, no_charging_trip)
    # 目的地のリストを取得
    destination_list = vehicle_trip['goalID'].unique().tolist()
    destination_list = [dest for dest in destination_list if dest <= 900000]
    destination_list.sort()
#   # 目的地ごとの旅行時間をOD別にまとめる
    trip_summary = {
                'destination': destination_list,
                'charging': [],
                'no_charging': [],
                'add_time_for_charging': []
                }
    # 目的地ごとに充電ありとなしの旅行時間を計算
    for destination in destination_list:
        od_charging = charging_trip[charging_trip['goalID'] == destination]
        od_no_charging = no_charging_trip[no_charging_trip['goalID'] == destination]
        
        od_charging_summary = od_charging.groupby('time_slot')['trip_time'].mean().reset_index()
        od_no_charging_summary = od_no_charging.groupby('time_slot')['trip_time'].mean().reset_index()
        
        trip_summary['charging'].append(od_charging_summary['trip_time'])
        trip_summary['no_charging'].append(od_no_charging_summary['trip_time'])
        trip_summary['add_time_for_charging'].append(od_charging_summary['trip_time'].mean() - od_no_charging_summary['trip_time'].mean())
    # データフレームに変換
    return pd.DataFrame(trip_summary)

def _add_vehicle_trip_columns(charging_trip: pd.DataFrame, no_charging_trip: pd.DataFrame)-> tuple:
    """充電した車両と充電しなかった車両の旅行時間と時間帯を追加する関数"""
    time = pd.to_datetime(charging_trip['StartTime'], unit='s')
    charging_trip['trip_time'] = (charging_trip['EndTime'] - charging_trip['StartTime'])/ 60  # 分単位に変換
    charging_trip['time_slot'] = pd.cut(time.dt.hour, bins=np.arange(0, 25, 1), right=False)
    
    time_no_charging = pd.to_datetime(no_charging_trip['StartTime'], unit='s')
    no_charging_trip['trip_time'] = (no_charging_trip['EndTime'] - no_charging_trip['StartTime']) / 60  # 分単位に変換
    no_charging_trip['time_slot'] = pd.cut(time_no_charging.dt.hour, bins=np.arange(0, 25, 1), right=False)
    
    return charging_trip, no_charging_trip
    

In [ ]:
# データの読み込みと処理
def aggregate_results(data_files, data_path, num_files) -> pd.DataFrame:
    """過去のシミュレーション結果を集計する関数"""
    results = {"trial":[],
            "total_cost":[],
            "total_trip_time": [],
            "total_charging_energy": [],
            "initial_cost": [],
            "running_costs_yearly": [],
            "add_time_for_charging": [],
            "loss_penalty_time": [],
            "num_chr_complete": [],
            "cs_config": []
            }
    # 各ファイルを読み込み、総コスト、総旅行時間、総充電量を計算
    for i, file in enumerate(data_files):
        if i == 200: break  # 200ファイルまで読み込む
        with open(os.path.join(data_path, file), 'rb') as f:
            data = pickle.load(f)
        
        total_cost, emates_result, total_trip_time, initial_costs, running_costs_yearly  = total_costs_yearly(os.path.join(data_path, file))
        # 総旅行時間の算出
        vehicle_trip = emates_result['vehicle_trip']
        no_charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] == 0]
        charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] > 0]
        num_chr_complete = len(charging_trip)  # 充電完了した車両の数
        # 
        trip_times_df = get_od_trip_times(vehicle_trip)
        timeseries_kw = emates_result['time_series_kw']
        total_charging_energy = timeseries_kw.sum().sum() / 60  # kWh単位に変換
        # 充電失敗ペナルティ
        loss_penalty_time = _loss_penalty_time(emates_result)
        # 充電による追加時間の計算：追加時間の細分化：CSまでの迂回時間＋充電待ち時間＋充電時間
        results["trial"].append(file)
        results["total_cost"].append(total_cost)
        results["total_trip_time"].append(total_trip_time)
        results["total_charging_energy"].append(total_charging_energy)
        results["initial_cost"].append(initial_costs)
        results["running_costs_yearly"].append(running_costs_yearly)
        results["add_time_for_charging"].append(trip_times_df['add_time_for_charging'])
        results["loss_penalty_time"].append(loss_penalty_time)
        results["num_chr_complete"].append(num_chr_complete)
        results["cs_config"].append(emates_result['cs_config'])
    # データフレームに変換
    results_df = pd.DataFrame(results)
    return results_df

def _loss_penalty_time(emates_result: dict) -> float:
    """充電失敗ペナルティを計算する関数"""
    NEEDED_CHARGING_KWH = 36  # 充電に必要なkWh
    charging_loss = emates_result['charging_loss']
    num_loss = len(charging_loss)
    min_kw = min(emates_result['cs_config']['cap_kw'])
    loss_penalty_time = NEEDED_CHARGING_KWH * num_loss / min_kw * 60 * 2 # 分単位に変換&自分の充電時間＋充電待ち時間を考慮して二倍する。
    # 確実に諦めをペナルティとして学習させるために10%のマージンを取る。
    loss_penalty_time *= 1.1
    return loss_penalty_time

In [ ]:
# 可視化
def vis1(results_df):
    """過去データの指標の変化を可視化する関数"""
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 2, 1)
    plt.plot(results_df.index, results_df['total_trip_time'], marker='o', label='総旅行時間 (時間)')
    plt.title('総旅行時間の変化')
    plt.xlabel('Trial Index')
    plt.ylabel('総旅行時間 (時間)')
    plt.grid()
    
    plt.subplot(2, 2, 2)
    plt.plot(results_df.index, results_df['total_charging_energy'], marker='o', label='総充電量 (kWh)', color='orange')
    plt.title('総充電量の変化')
    plt.xlabel('Trial Index')
    plt.ylabel('総充電量 (kWh)')
    plt.grid()
    
    plt.subplot(2, 2, 3)
    plt.plot(results_df.index, results_df['initial_cost'], marker='o', label='初期コスト (万円)', color='green')
    plt.title('初期コストの変化')
    plt.xlabel('Trial Index')
    plt.ylabel('初期コスト (万円)')
    plt.grid()
    
    # 総コストのプロット
    plt.subplot(2, 2, 4)
    plt.plot(results_df.index, results_df['total_cost'], marker='o', label='総コスト (万円)', color='red')
    plt.title('総コストの変化')
    plt.xlabel('Trial Index')
    plt.ylabel('総コスト (万円)')

def vis2(results_df):
    """過去データの指標の変化を可視化する関数"""
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    mean_add_time = results_df['add_time_for_charging'].apply(lambda x: x.mean() if hasattr(x, 'mean') else float('nan'))
    sum_add_time = mean_add_time * results_df['num_chr_complete']  # 充電完了した車両の数で重み付け
    plt.plot(results_df.index, sum_add_time, marker='o', color='blue', alpha=0.5, linestyle='None')  # 点を重ねて表示
    # plt.plot(mean_add_time, marker='o', color='blue', alpha=0.5, linestyle='None')  # 点を重ねて表示
    plt.title('充電による追加時間の変化')
    plt.xlabel('Trial Index')
    plt.ylabel('充電による追加時間 (分)')
    plt.grid()
    
    # 充電失敗ペナルティのプロット
    plt.subplot(1, 2, 2)
    plt.plot(results_df.index, results_df['loss_penalty_time'], marker='o', label='充電失敗ペナルティ (分)', color='pink')
    plt.title('充電失敗ペナルティの変化')
    plt.xlabel('Trial Index')
    plt.ylabel('充電失敗ペナルティ (分)')

In [ ]:
# メイン処理
# 設定の読み込み
data_files, data_path, num_files = setting()
# 過去のシミュレーション結果を集計
results_df = aggregate_results(data_files, data_path, num_files)


In [ ]:
# 結果の可視化
vis1(results_df)
vis2(results_df)


In [ ]:
# 充電のための追加時間を可視化（箱ひげ図で）
plt.figure(figsize=(12, 6))
plt.subplot(1, 1, 1)
plt.boxplot(results_df['add_time_for_charging'].tolist(), vert=True, patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='blue'))
# 各試行ごとの追加時間の平均値を計算してプロット
mean_add_time = results_df['add_time_for_charging'].apply(lambda x: x.mean() if hasattr(x, 'mean') else float('nan'))
plt.plot(mean_add_time, marker='o', color='blue', alpha=0.5, linestyle='None')  # 点を重ねて表示
plt.title('充電による追加時間の分布')
plt.xlabel('Trial Index')
plt.ylabel('充電による追加時間 (分)')
plt.tight_layout()

In [ ]:
# データ数が少なかったので，充電した車両全体の総旅行時間を可視化

no_charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] == 0]
charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] > 0]

time = pd.to_datetime(charging_trip['StartTime'], unit='s')
charging_trip['trip_time'] = (charging_trip['EndTime'] - charging_trip['StartTime'])/ 60  # 分単位に変換
charging_trip['time_slot'] = pd.cut(time.dt.hour, bins=np.arange(0, 25, 1), right=False)
charging_trip_goal_summary = charging_trip.groupby(['time_slot', 'goalID'])['trip_time'].mean().reset_index()

time_no_charging = pd.to_datetime(no_charging_trip['StartTime'], unit='s')
no_charging_trip['trip_time'] = (no_charging_trip['EndTime'] - no_charging_trip['StartTime']) / 60  # 分単位に変換
no_charging_trip['time_slot'] = pd.cut(time_no_charging.dt.hour, bins=np.arange(0, 25, 1), right=False)
no_charging_trip_goal_summary = no_charging_trip.groupby(['time_slot', 'goalID'])['trip_time'].mean().reset_index()

# 時間帯別の旅行時間をリスト形式で準備(箱ひげ図，個々の車両旅行時間を確認したい用)
# time_slot_groups = charging_trip.groupby('time_slot')['trip_time'].apply(list)
# time_slot_groups_no_charging = no_charging_trip.groupby('time_slot')['trip_time'].apply(list)

# 時間帯別の充電時間を折れ線グラフで充電，非充電車両の旅行時間を比較
plt.figure(figsize=(12, 6))
plt.bar(charging_trip_goal_summary['time_slot'].astype(str), charging_trip_goal_summary['trip_time'], color='skyblue', label='充電車両の平均旅行時間 (分)')
plt.bar(no_charging_trip_goal_summary['time_slot'].astype(str), no_charging_trip_goal_summary['trip_time'], alpha=0.7, color='orange', label='非充電車両の平均旅行時間 (分)')
plt.title('時間帯別の平均旅行時間の比較')
plt.xlabel('時間帯')
plt.ylabel('平均旅行時間 (分)')
plt.xticks(rotation=45)
plt.grid()
plt.legend()


In [ ]:
# 充電による追加時間だけでは，充電を諦めた車両の影響の考慮ができていない。
# 充電失敗した車両の数をカウントし，その分ペナルティを加える。
# 充電失敗した車両の数をカウント
NEEDED_CHARGING_KWH = 36  # 必要な充電量(kWh).20->80%まで充電する場合の値
data_files, data_path, num_files = setting()
data_file = data_files[55]  # 最初のファイルを使用
with open(os.path.join(data_path, data_file), 'rb') as f:
    emates_result = pickle.load(f)
vehicle_trip = emates_result['vehicle_trip']
charging_loss = emates_result['charging_loss']
num_loss = len(charging_loss)
min_kw = min(emates_result['cs_config']['cap_kw'])
loss_penalty_time = NEEDED_CHARGING_KWH * num_loss / min_kw * 60 * 2 # 分単位に変換&自分の充電時間＋充電待ち時間を考慮して二倍する。
# 確実に諦めをペナルティとして学習させるために10%のマージンを取る。
loss_penalty_time *= 1.1
print(f"充電失敗による追加時間: {loss_penalty_time} 分")

In [ ]:
# 目的関数の指標の正規化。ここでは，パイロットランニングの結果を用いて、充電による追加時間を正規化する。
# 最初の50トライアル分を使用
data_files, data_path, num_files = setting()
data_files = data_files[:50]  # 最初の50ファイルを使用
results_df = aggregate_results(data_files, data_path, num_files)


### 並列処理

In [ ]:
import sys
import os
from datetime import datetime
import json
import pickle
import numpy as np
import pandas as pd
import optuna
from tqdm import tqdm
import matplotlib.pyplot as plt
import japanize_matplotlib
import warnings
warnings.filterwarnings("ignore")


# プロジェクトのルートディレクトリをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.simulation.run_emates import only_run_emates
from src.simulation.data_load import save_data_to_pickle
from src.util.path_manager import get_paths



import concurrent.futures
import threading
import time
from src.simulation.create_emates_env import prepare_parallel_environment

current_time = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = current_time
# SAVE_DIR = '20250804_1716'
os.makedirs(SAVE_DIR, exist_ok=True)
FAILURE_FLAG = True

def total_costs_yearly(result_file) -> tuple:
    """年間の総コストを計算"""
    with open(result_file, 'rb') as f:
        emates_result = pickle.load(f)
    timeseries_kw = emates_result['time_series_kw']
    cs_config = emates_result['cs_config']
    initial_costs = _calc_initial_costs(cs_config, timeseries_kw)
    running_costs_yearly = _calc_running_costs(cs_config, timeseries_kw)
    total_costs = initial_costs + running_costs_yearly
    return total_costs.sum(), emates_result

def _calc_initial_costs(cs_config, timeseries_kw)-> np.ndarray:
    """初期コストの計算"""
    CHARGER_COST = {"50": 380, "90": 666, "100": 730}
    SUBSTATION_COST_PER_KW = 2
    INSTALLATION_COST = 250
    capacity_list = cs_config['cap_kw']
    ports_list = cs_config['ports']
    ports_array = np.array(ports_list)
    capacity_array = np.array(capacity_list)
    max_capacity = ports_array * capacity_array
    charger_costs = np.array([CHARGER_COST[str(c)] for c in capacity_list])
    cs_costs = charger_costs * ports_array
    substation_costs = (SUBSTATION_COST_PER_KW * max_capacity).reshape(-1)
    installation_costs = np.array([INSTALLATION_COST] * len(ports_list))
    initial_costs = cs_costs + substation_costs + installation_costs
    return initial_costs

def _calc_running_costs(cs_config, timeseries_kw)-> np.ndarray:
    """年間のランニングコストの計算"""
    MAINTENANCE_COST_PER_YEAR = 30
    CONTRACT_COST_PER_KW_MONTH = 1911e-4
    USAGE_COST_PER_KWH_MONTH = 18e-4
    CHARGING_PRICE_PER_KWH = 50e-4
    ports_array = np.array(cs_config['ports'])
    capacity_array = np.array(cs_config['cap_kw'])
    max_capacity = ports_array * capacity_array
    maintainance_costs = np.array([MAINTENANCE_COST_PER_YEAR] * len(ports_array))
    contract_costs = CONTRACT_COST_PER_KW_MONTH * max_capacity.reshape(-1) * 12
    usage_costs = USAGE_COST_PER_KWH_MONTH * timeseries_kw.sum(axis=0) * 12
    charging_revenue = CHARGING_PRICE_PER_KWH * timeseries_kw.sum(axis=0) * 12
    running_costs_yearly = maintainance_costs + contract_costs + usage_costs - charging_revenue
    return running_costs_yearly

def cs_placement_objective(trial) -> float:
    """CS配置最適化の目的関数"""
    # 1.OptunaによるCS配置の提案
    cs_config = _set_cs_placement(trial)
    
            
    # 2. 1で設定したCS配置の妥当性チェック
    if not _check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf')
    
    #3. シミュレーション実行とコスト計算。evaluation_costを最小化したい！
    evaluation_cost, emates_results = simulate_and_calculate_cost(cs_config, trial)
    
    #4.ペナルティがあればここに追加...
    """
    # 最低限満たしてほしい条件：充電実現率80％以上の確保
    # 追記：2025/7/3
    # これは保障として入れていたが，目的関数に総旅行時間を加えることで
    # 実質的に満たされることが期待されることを仮定してコメントアウト
    
    HOPED_CHARGING_RATE = 0.8
    vehicle_trip = emates_results['vehicle_trip']
    charging_loss = emates_results['charging_loss']
    
    num_charging_loss = len(charging_loss)
    num_charging_trip = len(vehicle_trip[vehicle_trip['startChargingTime'] != 0])
    satisfied_charging_rate = (num_charging_trip - num_charging_loss) / num_charging_trip
    if satisfied_charging_rate < HOPED_CHARGING_RATE:
        print(f"充電実現率が低い: {satisfied_charging_rate:.2%}。ペナルティを適用します。")
        # 実現率に応じたペナルティを追加
        evaluation_cost += (HOPED_CHARGING_RATE - satisfied_charging_rate) * 1000
    """
    return evaluation_cost

def _check_cs_placement(cs_config: dict) -> bool:
    installed_locations = [i for i, port in enumerate(cs_config['ports']) if port > 0]
    if len(installed_locations) < 2:  # 最低2箇所は設置
        return False
    
    # ほかに制約があればここに追加
    # ....
    return True
    

def _set_cs_placement(trial)-> dict:
    paths = get_paths(1)  # worker_idは1と仮定
    # csList.txtの情報を取得
    cslist_file = paths['csList']
    with open(cslist_file, 'r') as f:
        cslist_data = f.readlines()
    cslist_data = [line.strip() for line in cslist_data if line.strip()]
    csids = [int(line.split(',')[0]) for line in cslist_data if line.strip()]
    ports_list = [int(line.split(',')[1]) for line in cslist_data if line.strip()]
    cap_kw_list = [int(line.split(',')[2]) for line in cslist_data if line.strip()]
    
    
    cs_config = {
        'csids': [],
        'ports': [],
        'cap_kw': []
    }
    for i, csid in enumerate(csids):
        # trialの最初は元々のデータを使用
        if trial.number == 0:
            ports = ports_list[i]
            capacity = cap_kw_list[i]
        else:        
            # 設置するかどうかを決定
            ports = trial.suggest_int(f'ports_{i}', 0, 4)
            if ports > 0:
                # 設置する場合のみ容量を決定
                capacity = trial.suggest_categorical(f'capacity_{i}', [50, 100])
            else:
                # 設置しない場合は容量は任意（50に固定）
                capacity = 90
        # ✅ 修正：辞書に追加
        
        cs_config['csids'].append(csid)
        cs_config['ports'].append(ports)
        cs_config['cap_kw'].append(capacity)
    return cs_config


def create_failure_scenario_parallel(cs_config: dict, trial, max_workers=None):
    """故障シナリオを並列で処理する統合関数"""
    
    # 1. 設置されているCSのインデックスを取得
    installed_cs_indices = [i for i, ports in enumerate(cs_config['ports']) if ports > 0]
    
    if len(installed_cs_indices) == 0:
        return float('inf'), float('inf')
    
    print(f"\n=== Trial {trial.number}: 故障シナリオ並列処理開始 ===")
    print(f"設置CS数: {len(installed_cs_indices)}, 並列シナリオ数: {len(installed_cs_indices)}")
    
    # 2. 並列数を決定（故障シナリオ数と同じ）
    parallel_count = len(installed_cs_indices)
    if max_workers:
        parallel_count = min(parallel_count, max_workers)
    
    # 3. 並列環境を構築（各ワーカーフォルダを作成）
    try:
        print("並列環境構築中...")
        prepare_parallel_environment(parallel_count, cs_config)
        print(f"✓ {parallel_count}個のワーカー環境を構築完了")
    except Exception as e:
        print(f"❌ 並列環境構築エラー: {e}")
        return float('inf'), float('inf')
    
    # 4. 各故障シナリオを並列で実行
    start_time = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=parallel_count) as executor:
        # 各故障シナリオのタスクを投入
        future_to_scenario = {}
        
        for worker_id, failure_cs_idx in enumerate(installed_cs_indices, 1):
            future = executor.submit(
                process_single_failure_scenario_with_worker,
                cs_config, failure_cs_idx, trial.number, SAVE_DIR, worker_id
            )
            future_to_scenario[future] = failure_cs_idx
            # print(f"故障シナリオCS{failure_cs_idx} → Worker{worker_id}に投入")
        
        # 結果を順次収集
        results = []
        completed = 0
        
        for future in concurrent.futures.as_completed(future_to_scenario):
            failure_cs_idx = future_to_scenario[future]
            
            try:
                result = future.result(timeout=3600)  # 1時間タイムアウト
                results.append(result)
                completed += 1
                
                print(f"✓ [{completed}/{len(installed_cs_indices)}] "
                      f"故障CS{failure_cs_idx}完了: "
                      f"コスト={result['cost']:.2f}万円, "
                      f"95%待ち時間={result['wait_time_95p']:.2f}秒")
                
            except concurrent.futures.TimeoutError:
                print(f"❌ 故障CS{failure_cs_idx}がタイムアウト")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': 'timeout'
                })
            except Exception as e:
                print(f"❌ 故障CS{failure_cs_idx}でエラー: {e}")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': str(e)
                })
    
    end_time = time.time()
    print(f"\n並列処理完了（{end_time - start_time:.2f}秒）")
    
    # 5. 結果の集計
    valid_results = [r for r in results if r['cost'] != float('inf')]
    
    if not valid_results:
        print("❌ 有効な結果がありません")
        return float('inf'), float('inf')
    
    # 最悪ケースの抽出
    worst_cost = max(r['cost'] for r in valid_results)
    worst_wait_time = max(r['wait_time_95p'] for r in valid_results)
    
    print(f"📊 結果サマリー:")
    print(f"  有効シナリオ数: {len(valid_results)}/{len(results)}")
    print(f"  最悪ケースコスト: {worst_cost:.2f}万円")
    print(f"  最悪ケース95%待ち時間: {worst_wait_time:.2f}秒")
    
    return worst_cost, worst_wait_time

def process_single_failure_scenario_with_worker(cs_config, failure_cs_idx, trial_number, save_dir, worker_id):
    """指定されたワーカーで単一の故障シナリオを処理"""
    
    try:
        
        # 1. 指定されたワーカー用に故障情報を作成
        _create_failure_info_for_worker(cs_config, failure_cs_idx, worker_id)
        
        # 2. 該当ワーカーでシミュレーション実行（単一ワーカーのみ）
        only_run_emates(worker_id=worker_id)
        
        # 3. 結果の保存
        results_filename = f"trial_{trial_number}_failure_cs_{failure_cs_idx}_worker_{worker_id}.pkl"
        results_filepath = os.path.join(save_dir, results_filename)
        save_data_to_pickle(worker_id=worker_id, filename=results_filepath)
        
        # 4. コスト計算
        failure_cost, _ = total_costs_yearly(result_file=results_filepath)
        
        # 5. 95パーセンタイル待ち時間計算
        wait_time_95p = calculate_95percentile_wait_time(results_filepath)
        
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': failure_cost,
            'wait_time_95p': wait_time_95p,
            'results_filepath': results_filepath
        }
        
    except Exception as e:
        print(f"❌ Worker{worker_id}で故障CS{failure_cs_idx}の処理中にエラー: {e}")
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': float('inf'),
            'wait_time_95p': float('inf'),
            'error': str(e)
        }

def calculate_95percentile_wait_time(result_file: str) -> float:
    """95パーセンタイル待ち時間を計算"""
    with open(result_file, 'rb') as f:
        emates_result = pickle.load(f)
    
    vehicle_trip = emates_result['vehicle_trip']
    
    # 充電を行ったトリップのみを抽出
    charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] != 0].copy()
    
    if len(charging_trip) == 0:
        return float('inf')  # 充電できなかった場合は無限大
    
    # 完了した充電のみを対象（EndTimeがNaNでない）
    completed_charging = charging_trip[~charging_trip['EndTime'].isna()].copy()
    
    if len(completed_charging) == 0:
        return float('inf')  # 完了した充電がない場合は無限大
    
    # 待ち時間を計算
    waiting_times = (completed_charging['startChargingTime'].values - 
                    completed_charging['WaitingEntryTime'].values)
    
    # 95パーセンタイル待ち時間を計算
    wait_time_95p = np.percentile(waiting_times, 95)
    
    return wait_time_95p

def _create_failure_info_for_worker(cs_config: dict, failure_cs_index: int, worker_id: int) -> None:
    """指定されたワーカー用に故障情報を作成"""
    time_interval = 3600  # 1時間ごと
    max_time = 24 * 3600  # 24時間
    failure_time = 3600 * 12  # 12時に故障
    
    # 指定されたワーカーのパスを取得
    paths = get_paths(worker_id)
    OPENDSS_PATH = f'{paths["result"]}/opendss'
    
    # opendssディレクトリが存在しない場合は作成
    os.makedirs(OPENDSS_PATH, exist_ok=True)
    
    var_names = ["Type", "Node", "Csid", "Chgrid", "kW_0min", 
                "kW_60min", "kW_120min", "Yen_0min", "Yen_60min", "Yen_120min"]
    
    for time in range(time_interval, max_time + time_interval, time_interval):
        rows = []
        
        for cs_idx, (csid, ports, cap_kw) in enumerate(zip(cs_config['csids'], cs_config['ports'], cs_config['cap_kw'])):
            if ports == 0:  # 設置されていないCSはスキップ
                continue
            # csvファイルの行を作成 
            for chgrid in range(ports):
                # 故障時間かつ指定されたCSの場合
                if time == failure_time and cs_idx == failure_cs_index:
                    # 故障CSは出力を0に設定
                    row = [
                        "F",         # Type
                        "Any",       # Node
                        csid,        # Csid
                        chgrid,      # Chgrid
                        0,           # kW_0min (故障のため0)
                        -1,          # kW_60min
                        -1,          # kW_120min
                        -1,          # Yen_0min
                        -1,          # Yen_60min
                        -1           # Yen_120min
                    ]
                else:
                    # 通常の出力設定
                    row = [
                        "F",         # Type
                        "Any",       # Node
                        csid,        # Csid
                        chgrid,      # Chgrid
                        cap_kw,      # kW_0min
                        -1,          # kW_60min
                        -1,          # kW_120min
                        -1,          # Yen_0min
                        -1,          # Yen_60min
                        -1           # Yen_120min
                    ]
                rows.append(row)
        
        # CSVファイルに保存
        E_filename = f"E{time+1:06d}"
        df = pd.DataFrame(rows, columns=var_names)
        save_Efile_path = f"{OPENDSS_PATH}/{E_filename}.csv"
        df.to_csv(save_Efile_path, index=False)

# 並列処理版の目的関数
def cs_placement_objective_failure_parallel(trial) -> tuple:
    """並列処理版：故障時のみを考慮した多目的最適化の目的関数"""
    
    # 1. OptunaによるCS配置の提案
    cs_config = _set_cs_placement(trial)
    
    # 2. CS配置の妥当性チェック
    if not _check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf'), float('inf')
    
    # 3. 並列故障シナリオ実行
    worst_failure_cost, worst_wait_time = create_failure_scenario_parallel(
        cs_config, trial, max_workers=8  # 最大4並列に制限
    )
    
    return worst_failure_cost, worst_wait_time

# 並列処理版の最適化実行
def run_optuna_failure_optimization_parallel(n_trials=50):
    """並列処理版故障時最適化の実行"""
    db_path = os.path.join(SAVE_DIR, 'optuna_failure_study_parallel.db')
    db_url = f"sqlite:///{db_path}"
    study_name = "cs_failure_optimization_parallel"
    
    try:
        study = optuna.load_study(study_name=study_name, storage=db_url)
        print(f"既存の並列処理Studyを読み込みました（トライアル数: {len(study.trials)}）")
    except KeyError:
        study = optuna.create_study(
            directions=['minimize', 'minimize'],
            study_name=study_name, 
            storage=db_url
        )
        print("新しい並列処理Studyを作成しました")
    
    print(f"\n🚀 並列故障シナリオ最適化を開始します")
    print(f"トライアル数: {n_trials}")
    print(f"各トライアルで故障シナリオを並列実行します")
    
    # 最適化実行
    # ✅ --- tqdmによるプログレスバーのセットアップ ---
    with tqdm(total=n_trials, desc="Optimization Progress") as pbar:
        # 各トライアル完了時にプログレスバーを更新するコールバック関数
        def progress_bar_callback(study, trial):
            pbar.update(1)
        study.optimize(cs_placement_objective_failure_parallel, n_trials=n_trials,
                       callbacks=[progress_bar_callback])
    # ✅ --- プログレスバーのセットアップここまで ---
    
    # 結果表示と保存
    visualize_failure_pareto_front(study)
    save_failure_optimization_results(study)
    
    return study

def visualize_failure_pareto_front(study):
    """故障時パレートフロントの可視化"""
    import matplotlib.pyplot as plt
    
    # 完了したトライアルのみを取得
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    
    if len(completed_trials) == 0:
        print("完了したトライアルがありません")
        return
    
    # 無限大値を除外
    finite_trials = [t for t in completed_trials if not any(v == float('inf') for v in t.values)]
    
    if len(finite_trials) == 0:
        print("有効な解がありません")
        return
    
    # 目的関数値を取得
    costs = [t.values[0] for t in finite_trials]
    wait_times = [t.values[1] for t in finite_trials]
    
    # パレート最適解を取得
    pareto_trials = study.best_trials
    pareto_costs = [t.values[0] for t in pareto_trials if not any(v == float('inf') for v in t.values)]
    pareto_wait_times = [t.values[1] for t in pareto_trials if not any(v == float('inf') for v in t.values)]
    
    # プロット
    plt.figure(figsize=(12, 8))
    plt.scatter(costs, wait_times, alpha=0.6, label='全解', color='lightblue')
    if pareto_costs and pareto_wait_times:
        plt.scatter(pareto_costs, pareto_wait_times, color='red', s=100, label='パレート最適解', zorder=5)
        
        # パレート最適解に番号を付ける
        for i, (cost, wait) in enumerate(zip(pareto_costs, pareto_wait_times)):
            plt.annotate(f'P{i}', (cost, wait), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8)
    
    plt.xlabel('故障時平均コスト (万円)')
    plt.ylabel('最悪ケース95%待ち時間 (秒)')
    plt.title('故障時のコスト vs 待ち時間パレートフロント')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # パレート最適解の詳細を表示
    print("\n=== 故障時パレート最適解 ===")
    valid_pareto_trials = [t for t in study.best_trials if not any(v == float('inf') for v in t.values)]
    for i, trial in enumerate(valid_pareto_trials):
        print(f"解P{i}: 故障時平均コスト={trial.values[0]:.2f}万円, "
              f"最悪待ち時間={trial.values[1]:.2f}秒")
        print(f"  パラメータ: {trial.params}")

def save_failure_optimization_results(study):
    """故障時最適化結果の保存"""
    results_dir = os.path.join(SAVE_DIR, 'failure_optimization_results')
    os.makedirs(results_dir, exist_ok=True)
    
    # パレート最適解の保存（無限大値を除外）
    pareto_results = []
    valid_pareto_trials = [t for t in study.best_trials if not any(v == float('inf') for v in t.values)]
    
    for i, trial in enumerate(valid_pareto_trials):
        pareto_results.append({
            'solution_id': f'P{i}',
            'failure_avg_cost': trial.values[0],
            'worst_wait_time': trial.values[1],
            'params': trial.params,
            'trial_number': trial.number
        })
    
    with open(os.path.join(results_dir, 'failure_pareto_solutions.json'), 'w', encoding='utf-8') as f:
        json.dump(pareto_results, f, indent=2, ensure_ascii=False)
    
    # 全トライアルの結果も保存
    all_results = []
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE and not any(v == float('inf') for v in trial.values):
            all_results.append({
                'trial_number': trial.number,
                'failure_avg_cost': trial.values[0],
                'worst_wait_time': trial.values[1],
                'params': trial.params
            })
    
    with open(os.path.join(results_dir, 'all_failure_results.json'), 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    
    print(f"故障時パレート最適解を {results_dir}/failure_pareto_solutions.json に保存しました")
    print(f"全結果を {results_dir}/all_failure_results.json に保存しました")

# 実行例
print("=== 並列処理版故障シナリオ最適化 ===")
study_parallel = run_optuna_failure_optimization_parallel(n_trials=100)

### DBの読み込み


In [ ]:
import optuna
import japanize_matplotlib
import warnings
import pandas as pd
warnings.filterwarnings("ignore")
from optuna.visualization import plot_hypervolume_history
db_path = '20250806_1440/optuna_failure_study_parallel.db'
db_url = f"sqlite:///{db_path}"
study_name = "cs_failure_optimization_parallel"

studied_0806 = optuna.load_study(study_name=study_name, storage=db_url)
def visualize_failure_pareto_front(study):
    """故障時パレートフロントの可視化"""
    import matplotlib.pyplot as plt
    
    # 完了したトライアルのみを取得
    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    
    if len(completed_trials) == 0:
        print("完了したトライアルがありません")
        return
    
    # 無限大値を除外
    finite_trials = [t for t in completed_trials if not any(v == float('inf') for v in t.values)]
    
    if len(finite_trials) == 0:
        print("有効な解がありません")
        return
    
    # 目的関数値を取得
    costs = [t.values[0] for t in finite_trials]
    # 分に変換
    wait_times = [t.values[1] /60 for t in finite_trials]
    
    # パレート最適解を取得
    pareto_trials = study.best_trials
    pareto_costs = [t.values[0] for t in pareto_trials if not any(v == float('inf') for v in t.values)]
    pareto_wait_times = [t.values[1] /60 for t in pareto_trials if not any(v == float('inf') for v in t.values)]
    
    # プロット
    plt.figure(figsize=(12, 8))
    plt.scatter(costs, wait_times, alpha=0.6, label='全解', color='lightblue')
    if pareto_costs and pareto_wait_times:
        plt.scatter(pareto_costs, pareto_wait_times, color='red', s=100, label='パレート最適解', zorder=5)
        
        # パレート最適解に番号を付ける
        for i, (cost, wait) in enumerate(zip(pareto_costs, pareto_wait_times)):
            plt.annotate(f'P{i}', (cost, wait), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8)
    
    plt.xlabel('故障時平均コスト (万円)')
    plt.ylabel('最悪ケース95%待ち時間 (分)')
    plt.title('故障時のコスト vs 待ち時間パレートフロント')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # パレート最適解の詳細表示（改善版）
    print("\n" + "="*80)
    print("故障時パレート最適解の詳細")
    print("="*80)
    
    valid_pareto_trials = [t for t in study.best_trials if not any(v == float('inf') for v in t.values)]
    
    for i, trial in enumerate(valid_pareto_trials):
        print(f"\n【解P{i}】")
        print(f"  故障時平均コスト: {trial.values[0]:.2f}万円")
        print(f"  最悪待ち時間: {trial.values[1]/60:.2f}分")
        print(f"  トライアル番号: {trial.number}")
        
        # パラメータを見やすく整理
        print(f"  充電ステーション配置:")
        
        # CSIDとパラメータの対応を取得
        cs_data = []
        ports_params = {k: v for k, v in trial.params.items() if k.startswith('ports_')}
        capacity_params = {k: v for k, v in trial.params.items() if k.startswith('capacity_')}
        
        for param_name, ports in ports_params.items():
            cs_index = int(param_name.split('_')[1])
            capacity_key = f'capacity_{cs_index}'
            capacity = capacity_params.get(capacity_key, 'N/A')
            
            if ports > 0:  # 設置されているCSのみ表示
                cs_data.append({
                    'CS番号': cs_index,
                    'ポート数': ports,
                    '容量(kW)': capacity,
                    '総容量(kW)': ports * capacity if capacity != 'N/A' else 'N/A'
                })
        
        if cs_data:
            df = pd.DataFrame(cs_data)
            print(df.to_string(index=False, justify='center'))
            
            # サマリー情報
            total_ports = sum([cs['ポート数'] for cs in cs_data])
            total_capacity = sum([cs['総容量(kW)'] for cs in cs_data if cs['総容量(kW)'] != 'N/A'])
            installed_cs_count = len(cs_data)
            
            print(f"\n  📊 サマリー:")
            print(f"    設置CS数: {installed_cs_count}箇所")
            print(f"    総ポート数: {total_ports}ポート")
            print(f"    総容量: {total_capacity}kW")
        else:
            print("    設置されているCSがありません")
        
        print("-" * 80)

        
visualize_failure_pareto_front(studied_0806)
# 参照点を設定（各目的関数の最大値より大きな値を設定）
# 完了したトライアルから最大値を取得
completed_trials = [t for t in studied_0806.trials if t.state == optuna.trial.TrialState.COMPLETE]
finite_trials = [t for t in completed_trials if not any(v == float('inf') for v in t.values)]

if finite_trials:
    max_cost = max(t.values[0] for t in finite_trials)
    max_wait_time = max(t.values[1] for t in finite_trials)
    
    # 参照点は最大値より少し大きく設定
    reference_point = [max_cost * 1.1, max_wait_time * 1.1]
    
    fig = plot_hypervolume_history(studied_0806, reference_point=reference_point)
    display(fig)
else:
    print("有効なトライアルがありません")

In [ ]:
db_path = '20250804_1716/optuna_failure_study_parallel.db'
db_url = f"sqlite:///{db_path}"
study_name = "cs_failure_optimization_parallel"

studied_0804 = optuna.load_study(study_name=study_name, storage=db_url)
visualize_failure_pareto_front(studied_0804)

from optuna.visualization import plot_hypervolume_history

# 参照点を設定（各目的関数の最大値より大きな値を設定）
# 完了したトライアルから最大値を取得
completed_trials = [t for t in studied.trials if t.state == optuna.trial.TrialState.COMPLETE]
finite_trials = [t for t in completed_trials if not any(v == float('inf') for v in t.values)]

if finite_trials:
    max_cost = max(t.values[0] for t in finite_trials)
    max_wait_time = max(t.values[1] for t in finite_trials)
    
    # 参照点は最大値より少し大きく設定
    reference_point = [max_cost * 1.1, max_wait_time * 1.1]
    
    fig = plot_hypervolume_history(studied_0804, reference_point=reference_point)
    display(fig)
else:
    print("有効なトライアルがありません")

### 8/7：初期コストと待ち時間によるロバスト最適化｜通常時の最適化とそれが故障した際のパフォーマンスの比較
充電収益を含む運用コストは少なくとも年間の運用を考慮する必要性があるが本研究ではある断面のみを考慮する。（年スケールでの検討はEV普及率の変動を考慮するべきであり，今回は不十分であるため。）


In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.util.optimization import *

def cs_placement_objective(trial) -> float:
    """CS配置最適化の目的関数"""
    # 1.OptunaによるCS配置の提案
    cs_config = set_cs_placement(trial)
    
            
    # 2. 1で設定したCS配置の妥当性チェック
    if not check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf')
    
    #3. シミュレーション実行とコスト計算。evaluation_costを最小化したい！
    evaluation_cost, emates_results = evaluation_total_costs(cs_config, trial)
    
    #4.ペナルティがあればここに追加...
    """
    # 最低限満たしてほしい条件：充電実現率80％以上の確保
    # 追記：2025/7/3
    # これは保障として入れていたが，目的関数に総旅行時間を加えることで
    # 実質的に満たされることが期待されることを仮定してコメントアウト
    
    HOPED_CHARGING_RATE = 0.8
    vehicle_trip = emates_results['vehicle_trip']
    charging_loss = emates_results['charging_loss']
    
    num_charging_loss = len(charging_loss)
    num_charging_trip = len(vehicle_trip[vehicle_trip['startChargingTime'] != 0])
    satisfied_charging_rate = (num_charging_trip - num_charging_loss) / num_charging_trip
    if satisfied_charging_rate < HOPED_CHARGING_RATE:
        print(f"充電実現率が低い: {satisfied_charging_rate:.2%}。ペナルティを適用します。")
        # 実現率に応じたペナルティを追加
        evaluation_cost += (HOPED_CHARGING_RATE - satisfied_charging_rate) * 1000
    """
    return evaluation_cost

def create_failure_scenario_parallel(cs_config: dict, trial, max_workers=None):
    """故障シナリオを並列で処理する統合関数"""
    
    # 1. 設置されているCSのインデックスを取得
    installed_cs_indices = [i for i, ports in enumerate(cs_config['ports']) if ports > 0]
    
    if len(installed_cs_indices) == 0:
        return float('inf'), float('inf')
    
    print(f"\n=== Trial {trial.number}: 故障シナリオ並列処理開始 ===")
    print(f"設置CS数: {len(installed_cs_indices)}, 並列シナリオ数: {len(installed_cs_indices)}")
    
    # 2. 並列数を決定（故障シナリオ数と同じ）
    parallel_count = len(installed_cs_indices)
    if max_workers:
        parallel_count = min(parallel_count, max_workers)
    
    # 3. 並列環境を構築（各ワーカーフォルダを作成）
    try:
        print("並列環境構築中...")
        prepare_parallel_environment(cs_config, parallel_count)
        print(f"✓ {parallel_count}個のワーカー環境を構築完了")
    except Exception as e:
        print(f"❌ 並列環境構築エラー: {e}")
        return float('inf'), float('inf')
    
    # 4. 各故障シナリオを並列で実行
    start_time = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=parallel_count) as executor:
        # 各故障シナリオのタスクを投入
        future_to_scenario = {}
        
        for worker_id, failure_cs_idx in enumerate(installed_cs_indices, 1):
            future = executor.submit(
                process_single_failure_scenario_with_worker,
                cs_config, failure_cs_idx, trial.number, SAVE_DIR, worker_id
            )
            future_to_scenario[future] = failure_cs_idx
            # print(f"故障シナリオCS{failure_cs_idx} → Worker{worker_id}に投入")
        
        # 結果を順次収集
        results = []
        completed = 0
        
        for future in concurrent.futures.as_completed(future_to_scenario):
            failure_cs_idx = future_to_scenario[future]
            
            try:
                result = future.result(timeout=3600)  # 1時間タイムアウト
                results.append(result)
                completed += 1
                
                print(f"✓ [{completed}/{len(installed_cs_indices)}] "
                      f"故障CS{failure_cs_idx}完了: "
                      f"コスト={result['cost']:.2f}万円, "
                      f"95%待ち時間={result['wait_time_95p']:.2f}秒")
                
            except concurrent.futures.TimeoutError:
                print(f"❌ 故障CS{failure_cs_idx}がタイムアウト")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': 'timeout'
                })
            except Exception as e:
                print(f"❌ 故障CS{failure_cs_idx}でエラー: {e}")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': str(e)
                })
    
    end_time = time.time()
    print(f"\n並列処理完了（{end_time - start_time:.2f}秒）")
    
    # 5. 結果の集計
    valid_results = [r for r in results if r['cost'] != float('inf')]
    
    if not valid_results:
        print("❌ 有効な結果がありません")
        return float('inf'), float('inf')
    
    # 最悪ケースの抽出
    worst_cost = max(r['cost'] for r in valid_results)
    worst_wait_time = max(r['wait_time_95p'] for r in valid_results)
    
    print(f"📊 結果サマリー:")
    print(f"  有効シナリオ数: {len(valid_results)}/{len(results)}")
    print(f"  最悪ケースコスト: {worst_cost:.2f}万円")
    print(f"  最悪ケース95%待ち時間: {worst_wait_time:.2f}秒")
    
    return worst_cost, worst_wait_time

def process_single_failure_scenario_with_worker(cs_config, failure_cs_idx, trial_number, save_dir, worker_id):
    """指定されたワーカーで単一の故障シナリオを処理"""
    
    try:
        
        # 1. 指定されたワーカー用に故障情報を作成
        _create_failure_info_for_worker(cs_config, failure_cs_idx, worker_id)
        
        # 2. 該当ワーカーでシミュレーション実行（単一ワーカーのみ）
        only_run_emates(worker_id=worker_id)
        
        # 3. 結果の保存
        results_filename = f"trial_{trial_number}_failure_cs_{failure_cs_idx}_worker_{worker_id}.pkl"
        results_filepath = os.path.join(save_dir, results_filename)
        save_data_to_pickle(worker_id=worker_id, filename=results_filepath)
        
        # 4. コスト計算(初期コストを計算)
        init_cost = calc_initial_costs(cs_config)
        
        # 5. 95パーセンタイル待ち時間計算
        wait_time_95p = calculate_95percentile_wait_time(results_filepath)
        
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': init_cost,
            'wait_time_95p': wait_time_95p,
            'results_filepath': results_filepath
        }
        
    except Exception as e:
        print(f"❌ Worker{worker_id}で故障CS{failure_cs_idx}の処理中にエラー: {e}")
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': float('inf'),
            'wait_time_95p': float('inf'),
            'error': str(e)
        }

def calculate_95percentile_wait_time(result_file: str) -> float:
    """95パーセンタイル待ち時間を計算"""
    with open(result_file, 'rb') as f:
        emates_result = pickle.load(f)
    
    vehicle_trip = emates_result['vehicle_trip']
    
    # 充電を行ったトリップのみを抽出
    charging_trip = vehicle_trip[vehicle_trip['startChargingTime'] != 0].copy()
    
    if len(charging_trip) == 0:
        return float('inf')  # 充電できなかった場合は無限大
    
    # 完了した充電のみを対象（EndTimeがNaNでない）
    completed_charging = charging_trip[~charging_trip['EndTime'].isna()].copy()
    
    if len(completed_charging) == 0:
        return float('inf')  # 完了した充電がない場合は無限大
    
    # 待ち時間を計算
    waiting_times = (completed_charging['startChargingTime'].values - 
                    completed_charging['WaitingEntryTime'].values)
    
    # 95パーセンタイル待ち時間を計算
    wait_time_95p = np.percentile(waiting_times, 95)
    
    return wait_time_95p

def _create_failure_info_for_worker(cs_config: dict, failure_cs_index: int, worker_id: int) -> None:
    """指定されたワーカー用に故障情報を作成"""
    time_interval = 3600  # 1時間ごと
    max_time = 24 * 3600  # 24時間
    failure_time = 3600 * 12  # 12時に故障
    
    # 指定されたワーカーのパスを取得
    paths = get_paths(worker_id)
    OPENDSS_PATH = f'{paths["result"]}/opendss'
    
    # opendssディレクトリが存在しない場合は作成
    os.makedirs(OPENDSS_PATH, exist_ok=True)
    
    var_names = ["Type", "Node", "Csid", "Chgrid", "kW_0min", 
                "kW_60min", "kW_120min", "Yen_0min", "Yen_60min", "Yen_120min"]
    
    for time in range(time_interval, max_time + time_interval, time_interval):
        rows = []
        
        for cs_idx, (csid, ports, cap_kw) in enumerate(zip(cs_config['csids'], cs_config['ports'], cs_config['cap_kw'])):
            if ports == 0:  # 設置されていないCSはスキップ
                continue
            # csvファイルの行を作成 
            for chgrid in range(ports):
                # 故障時間かつ指定されたCSの場合
                if time == failure_time and cs_idx == failure_cs_index:
                    # 故障CSは出力を0に設定
                    row = [
                        "F",         # Type
                        "Any",       # Node
                        csid,        # Csid
                        chgrid,      # Chgrid
                        0,           # kW_0min (故障のため0)
                        -1,          # kW_60min
                        -1,          # kW_120min
                        -1,          # Yen_0min
                        -1,          # Yen_60min
                        -1           # Yen_120min
                    ]
                else:
                    # 通常の出力設定
                    row = [
                        "F",         # Type
                        "Any",       # Node
                        csid,        # Csid
                        chgrid,      # Chgrid
                        cap_kw,      # kW_0min
                        -1,          # kW_60min
                        -1,          # kW_120min
                        -1,          # Yen_0min
                        -1,          # Yen_60min
                        -1           # Yen_120min
                    ]
                rows.append(row)
        
        # CSVファイルに保存
        E_filename = f"E{time+1:06d}"
        df = pd.DataFrame(rows, columns=var_names)
        save_Efile_path = f"{OPENDSS_PATH}/{E_filename}.csv"
        df.to_csv(save_Efile_path, index=False)
    
    # print(f"📝 Worker{worker_id}: 故障CS{failure_cs_index}の故障情報ファイル作成完了")

# 並列処理版の目的関数
def cs_placement_objective_failure_parallel(trial) -> tuple:
    """並列処理版：故障時のみを考慮した多目的最適化の目的関数"""
    
    # 1. OptunaによるCS配置の提案
    cs_config = _set_cs_placement(trial)
    
    # 2. CS配置の妥当性チェック
    if not _check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf'), float('inf')
    
    # 3. 並列故障シナリオ実行
    worst_failure_cost, worst_wait_time = create_failure_scenario_parallel(
        cs_config, trial, max_workers=8  # 最大4並列に制限
    )
    
    return worst_failure_cost, worst_wait_time

# 並列処理版の最適化実行
def run_optuna_failure_optimization_parallel(n_trials=50):
    """並列処理版故障時最適化の実行"""
    db_path = os.path.join(SAVE_DIR, 'optuna_failure_study_parallel.db')
    db_url = f"sqlite:///{db_path}"
    study_name = "cs_failure_optimization_parallel"
    
    try:
        study = optuna.load_study(study_name=study_name, storage=db_url)
        print(f"既存の並列処理Studyを読み込みました（トライアル数: {len(study.trials)}）")
    except KeyError:
        study = optuna.create_study(
            directions=['minimize', 'minimize'],
            study_name=study_name, 
            storage=db_url
        )
        print("新しい並列処理Studyを作成しました")
    
    print(f"\n🚀 並列故障シナリオ最適化を開始します")
    print(f"トライアル数: {n_trials}")
    print(f"各トライアルで故障シナリオを並列実行します")
    
    # 最適化実行
    # ✅ --- tqdmによるプログレスバーのセットアップ ---
    with tqdm(total=n_trials, desc="Optimization Progress") as pbar:
        # 各トライアル完了時にプログレスバーを更新するコールバック関数
        def progress_bar_callback(study, trial):
            pbar.update(1)
        study.optimize(cs_placement_objective_failure_parallel, n_trials=n_trials,
                       callbacks=[progress_bar_callback])
    # ✅ --- プログレスバーのセットアップここまで ---


### 単目的でコスト換算
- 通常時：= 初期コスト + sum(i=1~5, (運用コスト_通常時_i* + ユーザーコスト_通常時_i*365) / (1 + r)^i)

- 故障時：= 初期コスト + sum(i=1~5, (運用コスト_故障時_i* + ユーザーコスト_最悪故障時_i*365) / (1 + r)^i)

ユーザーコスト＝CSまでの移動による追加時間＋1.5 * 待ち時間

#### 通常時

In [ ]:
import os
import sys
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

SAVE_DIR = datetime.now().strftime('%Y%m%d_%H%M')
# SAVE_DIR = '20250804_1716'
os.makedirs(SAVE_DIR, exist_ok=True)
FAILURE_FLAG = True

# 初期コストの計算関数をインポート

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.util.optimization import *

def cs_placement_objective_normal(trial) ->float:
    """通常のCS配置最適化の目的関数"""
    
    # 1. OptunaによるCS配置の提案
    cs_config = _set_cs_placement(trial)
    
    # 2. CS配置の妥当性チェック
    if not _check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf')
    # 3. シミュレーション実行とコスト計算
    evaluation_cost, emates_results = _simulate_and_calculate_cost(cs_config, trial)
    # 4. ペナルティがあればここに追加...
    # ここでは特にペナルティは設定しない
    return evaluation_cost
def _simulate_and_calculate_cost(cs_config: dict, trial) -> tuple:
    """シミュレーション実行とコスト計算"""
    
    # 1. シミュレーション実行
    paths = get_paths(1)  # worker_idは1と仮定
    emates_result = run_emates_simulation(cs_config, paths, trial.number)
    
    # 2. 初期コストの計算
    initial_costs = calc_initial_costs(cs_config)
    
    # 3. 結果から評価コストを計算
    evaluation_cost = initial_costs + emates_result['total_cost']
    
    return evaluation_cost, emates_result

#### 故障時

In [ ]:
a = [1, 2 ,3]
for i in range(12):
    if i in a:
        print(i)
    else:
        print("not in")

In [ ]:
import sys
import os
from datetime import datetime
import json
import pickle
import numpy as np
import pandas as pd
import optuna
from tqdm import tqdm
import matplotlib.pyplot as plt
import japanize_matplotlib
import shutil
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# プロジェクトのルートディレクトリをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.simulation.run_emates import only_run_emates
from src.simulation.data_load import save_data_to_pickle
from src.util.path_manager import get_paths
from src.util.optimization import *

import concurrent.futures
import threading
import time
from src.simulation.create_emates_env import prepare_parallel_environment

current_time = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = current_time
SAVE_DIR = '20250816_1051_2hour_fail'
os.makedirs(SAVE_DIR, exist_ok=True)
FAILURE_FLAG = True

def create_failure_scenario_parallel(cs_config: dict, trial, max_workers=None):
    """故障シナリオを並列で処理する統合関数"""
    
    # 1. 設置されているCSのインデックスを取得
    installed_cs_indices = [i for i, ports in enumerate(cs_config['ports']) if ports > 0]
    
    if len(installed_cs_indices) == 0:
        return float('inf'), float('inf')
    
    print(f"\n=== Trial {trial.number}: 故障シナリオ並列処理開始 ===")
    print(f"設置CS数: {len(installed_cs_indices)}, 並列シナリオ数: {len(installed_cs_indices)}")
    
    # 2. 並列数を決定（故障シナリオ数と同じ）
    parallel_count = len(installed_cs_indices)
    if max_workers:
        parallel_count = min(parallel_count, max_workers)
    
    # 3. 並列環境を構築（各ワーカーフォルダを作成）
    print("並列環境構築中...")
    prepare_parallel_environment(cs_config, parallel_count)
    print(f"✓ {parallel_count}個のワーカー環境を構築完了")
    
    # 4. 各故障シナリオを並列で実行
    start_time = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=parallel_count) as executor:
        # 各故障シナリオのタスクを投入
        future_to_scenario = {}
        
        for worker_id, failure_cs_idx in enumerate(installed_cs_indices, 1):
            future = executor.submit(
                process_single_failure_scenario_with_worker,
                cs_config, failure_cs_idx, trial.number, SAVE_DIR, worker_id
            )
            future_to_scenario[future] = failure_cs_idx
            # print(f"故障シナリオCS{failure_cs_idx} → Worker{worker_id}に投入")
        
        # 結果を順次収集
        results = []
        completed = 0
        
        for future in concurrent.futures.as_completed(future_to_scenario):
            failure_cs_idx = future_to_scenario[future]
            
            try:
                result = future.result(timeout=3600)  # 1時間タイムアウト
                results.append(result)
                completed += 1
                
                print(f"✓ [{completed}/{len(installed_cs_indices)}] "
                      f"故障CS{failure_cs_idx}完了: "
                      f"コスト={result['cost']:.2f}万円, "
                      f"95%待ち時間={result['wait_time_95p']:.2f}秒")
                
            except concurrent.futures.TimeoutError:
                print(f"❌ 故障CS{failure_cs_idx}がタイムアウト")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': 'timeout'
                })
            except Exception as e:
                print(f"❌ 故障CS{failure_cs_idx}でエラー: {e}")
                results.append({
                    'failure_cs_idx': failure_cs_idx,
                    'cost': float('inf'),
                    'wait_time_95p': float('inf'),
                    'error': str(e)
                })
    
    end_time = time.time()
    print(f"\n並列処理完了（{end_time - start_time:.2f}秒）")
    
    # 5. 結果の集計
    valid_results = [r for r in results if r['cost'] != float('inf')]
    
    if not valid_results:
        print("❌ 有効な結果がありません")
        return float('inf')
    
    # 最悪ケースの抽出
    worst_cost = max(r['cost'] for r in valid_results)
    worst_wait_time = max(r['wait_time_95p'] for r in valid_results)
    
    print(f"📊 結果サマリー:")
    print(f"  有効シナリオ数: {len(valid_results)}/{len(results)}")
    print(f"  最悪ケースコスト: {worst_cost:.2f}万円")
    print(f"  最悪ケース95%待ち時間: {worst_wait_time:.2f}秒")
    
    return worst_cost

def process_single_failure_scenario_with_worker(cs_config, failure_cs_idx, trial_number, save_dir, worker_id):
    """指定されたワーカーで単一の故障シナリオを処理"""
    
    try:
        
        # 1. 指定されたワーカー用に故障情報を作成
        create_failure_info_for_worker(cs_config, failure_cs_idx, worker_id, FAILURE_TIME=[14, 15])
        
        # 2. 該当ワーカーでシミュレーション実行（単一ワーカーのみ）
        only_run_emates(worker_id=worker_id)
        
        # 3. 結果の保存
        results_filename = f"trial_{trial_number}_failure_cs_{failure_cs_idx}_worker_{worker_id}.pkl"
        results_filepath = os.path.join(save_dir, results_filename)
        save_data_to_pickle(worker_id=worker_id, filename=results_filepath)
        
        # 4. コスト計算
        eval_costs, _ = evaluation_total_costs(results_filepath)
        
        
        # 5. 95パーセンタイル待ち時間計算
        wait_time_95p = calculate_95percentile_wait_time(results_filepath)
        
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': eval_costs,
            'wait_time_95p': wait_time_95p,
            'results_filepath': results_filepath
        }
        
    except Exception as e:
        print(f"❌ Worker{worker_id}で故障CS{failure_cs_idx}の処理中にエラー: {e}")
        return {
            'failure_cs_idx': failure_cs_idx,
            'worker_id': worker_id,
            'cost': float('inf'),
            'wait_time_95p': float('inf'),
            'error': str(e)
        }

# 並列処理版の目的関数
def cs_placement_objective_failure_parallel(trial) -> tuple:
    """並列処理版：故障時のみを考慮した多目的最適化の目的関数"""
    
    # 1. OptunaによるCS配置の提案
    cs_config = set_cs_placement(trial)
    
    # 2. CS配置の妥当性チェック
    if not check_cs_placement(cs_config):
        print("CS配置が不適切です。最低2箇所の充電ステーションを設置してください。")
        return float('inf')
    
    # 3. 並列故障シナリオ実行
    worst_failure_cost = create_failure_scenario_parallel(
        cs_config, trial, max_workers=8  # 最大4並列に制限
    )
    
    return worst_failure_cost

def manage_pkl_files_after_optimization(study, save_dir):
    """最適化完了後のpklファイル管理"""
    print("\n=== 最適化完了後のファイル管理 ===")
    
    save_path = Path(save_dir)
    pkl_files = [f for f in os.listdir(save_path) if f.endswith('.pkl')]
    
    if not pkl_files:
        print("pklファイルが見つかりません。")
        return
    
    # ベストtrialを特定
    best_trial_number = study.best_trial.number
    print(f"ベストtrial: {best_trial_number}")
    
    # 保持するファイルを選定
    trials_to_keep = set()
    
    # 1. ベストtrial
    best_file = f"trial_{best_trial_number}.pkl"
    if best_file in pkl_files:
        trials_to_keep.add(best_file)
    
    # 2. 10の倍数のtrial
    for f in pkl_files:
        try:
            trial_num = int(f.split('_')[1].split('.')[0])
            if trial_num % 10 == 0:
                trials_to_keep.add(f)
        except (IndexError, ValueError):
            continue
    
    # 3. 最新10個のtrial
    try:
        sorted_files = sorted(pkl_files, 
                            key=lambda x: int(x.split('_')[1].split('.')[0]), 
                            reverse=True)
        for f in sorted_files[:10]:
            trials_to_keep.add(f)
    except (IndexError, ValueError):
        print("ファイル名の解析に失敗しました。")
    
    # 4. 削除対象を特定
    files_to_delete = set(pkl_files) - trials_to_keep
    
    # 統計情報を表示
    print(f"\n総ファイル数: {len(pkl_files)}")
    print(f"保持ファイル数: {len(trials_to_keep)}")
    print(f"削除対象ファイル数: {len(files_to_delete)}")
    
    if len(files_to_delete) == 0:
        print("削除対象のファイルはありません。")
        return
    
    # ファイルサイズの分析
    keep_size = 0
    delete_size = 0
    
    for file_name in trials_to_keep:
        file_path = save_path / file_name
        if file_path.exists():
            keep_size += file_path.stat().st_size
    
    for file_name in files_to_delete:
        file_path = save_path / file_name
        if file_path.exists():
            delete_size += file_path.stat().st_size
    
    print(f"保持データサイズ: {keep_size / (1024**2):.1f} MB")
    print(f"削除データサイズ: {delete_size / (1024**2):.1f} MB")
    print(f"削除による節約: {delete_size / (keep_size + delete_size) * 100:.1f}%")
    
    # 保持ファイルの内訳を表示
    print(f"\n=== 保持ファイル詳細 ===")
    for file in sorted(trials_to_keep):
        try:
            trial_num = int(file.split('_')[1].split('.')[0])
            file_path = save_path / file
            size_mb = file_path.stat().st_size / (1024**2) if file_path.exists() else 0
            
            reason = []
            if file == best_file:
                reason.append("ベスト")
            if trial_num % 10 == 0:
                reason.append("10の倍数")
            if file in sorted_files[:10]:
                reason.append("最新10個")
            
            print(f"  {file} (trial {trial_num}, {size_mb:.1f}MB) - {', '.join(reason)}")
        except (IndexError, ValueError):
            print(f"  {file} - ファイル名解析失敗")
    
    # 自動的にバックアップディレクトリに移動
    backup_dir = save_path / "backup_deleted_trials"
    backup_dir.mkdir(exist_ok=True)
    
    moved_count = 0
    for file_name in files_to_delete:
        file_path = save_path / file_name
        backup_path = backup_dir / file_name
        
        if file_path.exists():
            try:
                shutil.move(str(file_path), str(backup_path))
                moved_count += 1
            except Exception as e:
                print(f"ファイル移動失敗 {file_name}: {e}")
    
    print(f"\n{moved_count}個のファイルをバックアップディレクトリに移動しました。")
    print(f"バックアップ先: {backup_dir}")
    
    # 残りファイルサイズを確認
    remaining_size = sum(f.stat().st_size for f in save_path.glob("*.pkl")) / (1024**2)
    print(f"残りのpklファイルサイズ: {remaining_size:.1f} MB")

# 並列処理版の最適化実行
def run_optuna_failure_optimization_parallel(n_trials=150, timeout = 60*60*24):
    """並列処理版故障時最適化の実行"""
    db_path = os.path.join(SAVE_DIR, 'optuna_failure_study_parallel.db')
    db_url = f"sqlite:///{db_path}"
    study_name = "cs_failure_14_15hour"
    
    try:
        study = optuna.load_study(study_name=study_name, storage=db_url)
        print(f"既存の並列処理Studyを読み込みました（トライアル数: {len(study.trials)}）")
    except KeyError:
        study = optuna.create_study(
            directions=['minimize'],
            study_name=study_name, 
            storage=db_url
        )
        print("新しい並列処理Studyを作成しました")
    
    print(f"\n🚀 並列故障シナリオ最適化を開始します")
    print(f"トライアル数: {n_trials}")
    print(f"各トライアルで故障シナリオを並列実行します")
    
    # 最適化実行
    # ✅ --- tqdmによるプログレスバーのセットアップ ---
    with tqdm(total=n_trials, desc="Optimization Progress") as pbar:
        # 各トライアル完了時にプログレスバーを更新するコールバック関数
        def progress_bar_callback(study, trial):
            pbar.update(1)
        study.optimize(cs_placement_objective_failure_parallel, n_trials=n_trials,
                       callbacks=[progress_bar_callback], timeout=timeout)
    # ✅ --- プログレスバーのセットアップここまで ---
    manage_pkl_files_after_optimization(study, SAVE_DIR)


In [ ]:
run_optuna_failure_optimization_parallel(n_trials=100, timeout=60*60*10)  # 1日以内に完了するように設定

In [6]:
import optuna
import os
from pathlib import Path
import shutil
import pickle

SAVE_DIR = '20250815_1001_1hour_fail'
study = optuna.load_study(study_name="cs_failure_optimization_parallel", storage=f"sqlite:///{SAVE_DIR}/optuna_failure_study_parallel.db")

def manage_pkl_files_after_optimization(study, save_dir):
    """最適化完了後のpklファイル管理（ベストトライアルのみ保持）"""
    print("\n=== 最適化完了後のファイル管理（ベストのみ保持）===")
    
    save_path = Path(save_dir)
    pkl_files = [f for f in os.listdir(save_path) if f.endswith('.pkl')]
    
    if not pkl_files:
        print("pklファイルが見つかりません。")
        return
    
    # ベストtrialを特定
    best_trial_number = study.best_trial.number
    print(f"ベストtrial: {best_trial_number}")
    
    # 保持するファイルを選定（ベストのみ）
    trials_to_keep = set()
    
    # ベストtrialのファイルパターンを検索
    best_files = [f for f in pkl_files if f.startswith(f"trial_{best_trial_number}_") or f == f"trial_{best_trial_number}.pkl"]
    
    for best_file in best_files:
        trials_to_keep.add(best_file)
        print(f"保持ファイル: {best_file}")
    
    if len(trials_to_keep) == 0:
        print(f"⚠️  ベストtrial {best_trial_number} に対応するファイルが見つかりません。")
        print("利用可能なファイル:")
        for f in pkl_files[:10]:  # 最初の10個を表示
            print(f"  {f}")
        return
    
    # 削除対象を特定
    files_to_delete = set(pkl_files) - trials_to_keep
    
    # 統計情報を表示
    print(f"\n総ファイル数: {len(pkl_files)}")
    print(f"保持ファイル数: {len(trials_to_keep)}")
    print(f"削除対象ファイル数: {len(files_to_delete)}")
    
    if len(files_to_delete) == 0:
        print("削除対象のファイルはありません。")
        return
    
    # ファイルサイズの分析
    keep_size = 0
    delete_size = 0
    
    for file_name in trials_to_keep:
        file_path = save_path / file_name
        if file_path.exists():
            keep_size += file_path.stat().st_size
    
    for file_name in files_to_delete:
        file_path = save_path / file_name
        if file_path.exists():
            delete_size += file_path.stat().st_size
    
    print(f"保持データサイズ: {keep_size / (1024**2):.1f} MB")
    print(f"削除データサイズ: {delete_size / (1024**2):.1f} MB")
    print(f"削除による節約: {delete_size / (keep_size + delete_size) * 100:.1f}%")
    
    # 自動的にバックアップディレクトリに移動
    backup_dir = save_path / "backup_deleted_trials"
    backup_dir.mkdir(exist_ok=True)
    
    moved_count = 0
    for file_name in files_to_delete:
        file_path = save_path / file_name
        backup_path = backup_dir / file_name
        
        if file_path.exists():
            try:
                shutil.move(str(file_path), str(backup_path))
                moved_count += 1
            except Exception as e:
                print(f"ファイル移動失敗 {file_name}: {e}")
    
    print(f"\n✅ {moved_count}個のファイルをバックアップディレクトリに移動しました。")
    print(f"バックアップ先: {backup_dir}")
    print(f"💾 ベストトライアル {best_trial_number} のファイルのみ保持されました。")
    
    # 残りファイルサイズを確認
    remaining_size = sum(f.stat().st_size for f in save_path.glob("*.pkl")) / (1024**2)
    print(f"残りのpklファイルサイズ: {remaining_size:.1f} MB")
manage_pkl_files_after_optimization(study, SAVE_DIR)


=== 最適化完了後のファイル管理（ベストのみ保持）===
ベストtrial: 95
保持ファイル: trial_95_failure_cs_0_worker_1.pkl
保持ファイル: trial_95_failure_cs_1_worker_2.pkl
保持ファイル: trial_95_failure_cs_3_worker_3.pkl
保持ファイル: trial_95_failure_cs_5_worker_4.pkl

総ファイル数: 704
保持ファイル数: 4
削除対象ファイル数: 700
保持データサイズ: 20.9 MB
削除データサイズ: 3664.4 MB
削除による節約: 99.4%

✅ 700個のファイルをバックアップディレクトリに移動しました。
バックアップ先: 20250815_1001_1hour_fail\backup_deleted_trials
💾 ベストトライアル 95 のファイルのみ保持されました。
残りのpklファイルサイズ: 20.9 MB
